# Houston Weather EDA

This notebook walks through the portfolio rebuild of the Houston weather analysis project: data source, data fetching, cleaning, feature engineering, EDA summaries, correlation analysis, processed outputs, and generated visuals.

The original academic project scraped Wunderground daily history pages; that scraping portion was completed by a project teammate. My contribution focused on cleaning, EDA, visualization, interpretation, and reporting. This version uses Open-Meteo historical hourly weather data for reproducibility.

## 1. Data Source and Contribution Note

Source: Open-Meteo Historical Weather API  
URL: https://open-meteo.com/

Study window: `2022-01-01` to `2023-01-31`  
Location: Houston/KIAH-area coordinates, latitude `29.9844`, longitude `-95.3414`

Contribution note: the original Wunderground scraping in the team notebook was done by a teammate. This notebook highlights the cleaned portfolio rebuild and the EDA/interpretation workflow.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('..')
RAW_PATH = ROOT / 'data' / 'raw' / 'houston_hourly_weather_open_meteo_2022_2023.csv'
PROCESSED_DIR = ROOT / 'data' / 'processed'
FIGURES_DIR = ROOT / 'figures'

df = pd.read_csv(RAW_PATH)
df['datetime'] = pd.to_datetime(df['datetime'])
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['datetime'].dt.to_period('M').astype(str)
df.shape

In [ ]:
df.head()

## 2. Data Quality Checks

In [ ]:
numeric_cols = [
    'temperature_f', 'humidity_pct', 'dew_point_f', 'precipitation_in',
    'pressure_msl_hpa', 'wind_speed_mph', 'wind_gust_mph'
]

df[numeric_cols].isna().sum()

In [ ]:
df[numeric_cols].describe().T

## 3. Daily and Monthly Summaries

In [ ]:
daily = (
    df.groupby('date')
    .agg(
        mean_temp_f=('temperature_f', 'mean'),
        max_temp_f=('temperature_f', 'max'),
        min_temp_f=('temperature_f', 'min'),
        total_precip_in=('precipitation_in', 'sum'),
        mean_humidity_pct=('humidity_pct', 'mean'),
        max_wind_gust_mph=('wind_gust_mph', 'max'),
    )
    .reset_index()
)
daily.head()

In [ ]:
monthly = (
    df.groupby('month')
    .agg(
        avg_temp_f=('temperature_f', 'mean'),
        total_precip_in=('precipitation_in', 'sum'),
        avg_humidity_pct=('humidity_pct', 'mean'),
        avg_wind_speed_mph=('wind_speed_mph', 'mean'),
        max_wind_gust_mph=('wind_gust_mph', 'max'),
    )
    .reset_index()
)
monthly

## 4. Key Metrics

In [ ]:
corr = df[numeric_cols].corr().round(3)
hourly_temp = df.groupby('hour')['temperature_f'].mean().reset_index(name='avg_temp_f')

metrics = {
    'records': len(df),
    'days': df['date'].nunique(),
    'date_min': df['date'].min().strftime('%Y-%m-%d'),
    'date_max': df['date'].max().strftime('%Y-%m-%d'),
    'mean_temperature_f': df['temperature_f'].mean(),
    'max_temperature_f': df['temperature_f'].max(),
    'min_temperature_f': df['temperature_f'].min(),
    'total_precipitation_in': df['precipitation_in'].sum(),
    'rain_hours': int((df['precipitation_in'] > 0).sum()),
    'temp_dewpoint_corr': corr.loc['temperature_f', 'dew_point_f'],
    'temp_humidity_corr': corr.loc['temperature_f', 'humidity_pct'],
    'hottest_month': monthly.sort_values('avg_temp_f', ascending=False).iloc[0]['month'],
    'rainiest_day': daily.sort_values('total_precip_in', ascending=False).iloc[0]['date'].strftime('%Y-%m-%d'),
    'hottest_hour': int(hourly_temp.sort_values('avg_temp_f', ascending=False).iloc[0]['hour']),
}
metrics

## 5. Correlation Analysis

In [ ]:
corr

Temperature and dew point show a strong positive relationship, while temperature and relative humidity show a weak negative relationship.

## 6. Generated Outputs

The command-line script `src/analyze_weather.py` regenerates processed CSV files and SVG visuals.

In [ ]:
saved_metrics = pd.read_json(PROCESSED_DIR / 'metrics.json', typ='series')
saved_metrics

In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(FIGURES_DIR / 'kpi_summary.svg')))
display(SVG(filename=str(FIGURES_DIR / 'monthly_temperature.svg')))
display(SVG(filename=str(FIGURES_DIR / 'monthly_precipitation.svg')))
display(SVG(filename=str(FIGURES_DIR / 'correlation_heatmap.svg')))
display(SVG(filename=str(FIGURES_DIR / 'hourly_temperature_cycle.svg')))

## 7. Final Findings

- Houston weather showed a clear seasonal temperature cycle, with July 2022 as the hottest month.
- Temperature and dew point had a strong positive correlation of 0.839.
- Temperature and humidity had a weak negative correlation of -0.137.
- Rainfall was concentrated in fewer wetter periods, with January 24, 2023 as the rainiest day in this dataset.
- The warmest average hour was 3 PM.